<center>
    <p>105 Theoretical Neuroscience I</p>
    <h1></h1>
    <h1>Lecture 8:</h1>
    <h1>Neurons with more realistic morphology</h1>
    <p>----</p>
    <p>Prof. Jochen Braun Ph.D.</p>
    <p>Institute of Biology</p>
    <p>Otto-von-Guericke University Magdeburg</p>
    <p>----</p>
    <p>Textbook:</p>
    <p>Peter Dayan & Larry Abbott (2001) Theoretical Neuroscience, MIT Press.</p>
</center>

**Abstract:**

Today we introduce basic concepts for modelling the electric properties of neurons with a more realistic morphology. Considering cylindrical dendrite segments, we develop the **cable equation** and present some solutions for simple special cases, such as an **infinite cable** stimulated by a **constant current** or a brief **current pulse**. In dendritic trees with many branches, the electrical properties can be approximated by a single cylindrical segment (**equivalent dendrite**). Reconstructions of the dendritic trees of different cell types (**L2/3 pyramical, L5 Pyramidal, CA1 Pyramidal, Purkinj**e) allow detailed numerical simulations and the **morphoelectronic transform** strikingly illustrates the results. We conclude by modelling  **myelinated axons** to understand the multiple benefits of myelinization.

**Overview:**

1. Motivation
2. Cable equation
3. Some solutions of linear cable theory
4. Dendritic trees (optional)
5. Morphoelectronic transform
6. Myelinated axons (advanced)


# **1. Motivation**

Neurons with long and narrow processes cannot be treated as a single electronic compartment, as the membrane potential can vary considerably over the cell surface.  
<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Dendritic_spines_BMel.png" width="500">

Apical dendrites of pyramidal neurons with dendritic spines (enlarged). (Bartlett Mel, USC)


<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Neuron_morphology_BMel.png" width="500">

Examples of neuron morphology: pyramidal cell (A), Purkinje cell (B), stellate cell (C). (Bartlett Mel, USC)

(Slide-1)

# Conduction delays and attenuates time-course

Comparing a single membrane potential event (AP, EPSP) at distant sites (dendrite, soma) typically shows that the potential time-course is delayed and attenuated during conduction.

These effects are qualitatively, but not quantitatively, the same in the dendrite-to-soma and soma-to-dendrite direction.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08-D&A-Fig-6-5.png" width="800">

D&A Fig. 6-5

(Slide-2)

# Why does this matter?

Proximal synapses produce shorter, larger EPSPs.
Multiple shorter EPSPs must *coincide exactly* to have a cumulative effect.

Distal synapses produce longer, smaller EPSPs. Multiple longer EPSPs can have a cumulative effect even if they arrive *at slightly different times*.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Magee_2000_NRN.png" width="800">

[Magee (2000) Dendritic integration of excitatory synaptic inputs. Nature Review Neuroscience](https://www.nature.com/articles/35044552)

(Slide-3)


In [ ]:
# @title Dendritic integration of synaptic inputs {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact

# Illustrate dendritic integration

def DendriticEPSP( tau_fast, tau_slow, factor ):

    Ni = 1001;
    ti = np.linspace(0,10*tau_slow,Ni);

    P1_fast = np.zeros_like(ti);
    P1_slow = np.zeros_like(ti);

    dt = ti[1]-ti[0];

    P1_fast =     (ti/tau_fast) * np.exp( 1 - ti/tau_fast );
    P1_slow = 0.5*(ti/tau_slow) * np.exp( 1 - ti/tau_slow );

    Nfast   = round(7*tau_fast/dt);
    Nslow   = round(7*tau_slow/dt);

    fast    = np.linspace(0,Nfast,Nfast+1);
    slow    = np.linspace(0,Nslow,Nslow+1);
    ifast   = fast.astype(int);
    islow   = slow.astype(int);

    P5_fast = np.zeros_like(ti);
    P5_slow = np.zeros_like(ti);

    Nspike = 5;
    Noffset = round(factor*tau_fast/dt);

    for i in range(0,Nspike):
        fastoff = i*Noffset + ifast;
        slowoff = i*Noffset + islow;
        ifastoff = fastoff.astype(int);
        islowoff = slowoff.astype(int);

        P5_fast[ ifastoff[0:Ni] ] = P5_fast[ ifastoff[0:Ni] ] + P1_fast[ ifast[0:Ni] ];
        P5_slow[ islowoff[0:Ni] ] = P5_slow[ islowoff[0:Ni] ] + P1_slow[ islow[0:Ni] ];

    return ti, P1_fast, P1_slow, P5_fast, P5_slow

# Define an interactive plot
def plot_dendritic_integration(tau_fast = 5, tau_slow = 20, Offset = 2):
    """(Slide-4)"""

    # Generate and cumulate synaptic time-courses
    (ti, P1_fast, P1_slow, P5_fast, P5_slow) = DendriticEPSP( tau_fast, tau_slow, Offset );


    # Create a figure
    fig = plt.figure(figsize=(6, 4))

    # First subplot
    ax1 = fig.add_subplot(2, 1, 1)

    ax1.plot(ti, P1_fast, color='b', linewidth=0.5);
    ax1.plot(ti, P5_fast, color='b', linewidth=1);
    ax1.set_title('Fast EPSP', fontsize=14)
    ax1.set_ylabel('P(t)', fontsize=14)
    ax1.set_xlabel('time [ms]', fontsize=14)

    # Second subplot
    ax2 = fig.add_subplot(2, 1, 2)

    ax2.plot(ti, P1_slow, color='r', linewidth=0.5);
    ax2.plot(ti, P5_slow, color='r', linewidth=1);
    ax2.set_title('Slow EPSP', fontsize=14)
    ax2.set_ylabel('P(t)', fontsize=14)
    ax2.set_xlabel('time [ms]', fontsize=14)

    # Adjust layout to prevent overlap
    plt.tight_layout()

    # Display the plots
    plt.show()


# Creating sliders for concentrations and charge
interact(plot_dendritic_integration, tau_fast=(4,8,1), tau_slow=(10,40,10), Offset=(1.0,3.0,0.1) );



interactive(children=(IntSlider(value=5, description='tau_fast', max=8, min=4), IntSlider(value=20, descriptio…

# Summary motivation

- We wish to consider the functional consequences of neuron morphology, in particular of complex dendritic trees with 'passive' membranes.

- Conduction along `passive' dendrites typically delays and attenuates an electric signal (e.g., PSP, AP).

- This matters in the dendrite-to-soma direction, because electric delays, attenuation, summation and cancellation of PSPs determines exactly when a neuron reaches threshold and fires.

- It also matters in the soma-to-dendrite direction, because electric delays and attenuation of APs determines the postsynaptic potential at synaptic spines, which in turn governs synaptic plasticity (e.g., NMDA).

(Slide-5)

# **2. Cable theory**

The effects of passive conduction are most pronounced for long and narrow dendritic branches, which may be treated like an electric 'cable'.

The typical problem of ''cable theory'' (as formalized by [Wilfrid Rall, ca. 1957](https://direct.mit.edu/books/oa-edited-volume/2066/The-Theoretical-Foundation-of-Dendritic)) is to determine the distribution of voltage $V(x,t)$ along the length of the cable $x$ and over time $t$, given a current impulse $I_0(x_0,t_0)$ at position $x_0$ and time $t_0$.  


<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Cable_theory_A.png" width="600">

(Slide-6)




# Longitudinal resistance

Consider a cylindrical segment of length $\Delta x$, radius $a$, and cross-section $\pi\,a^2$.

The longitudinal resistance $R_L$ of this segment is

$\begin{eqnarray}
\qquad\qquad R_L = \frac{r_L\, \Delta x}{\pi\, a^2}
\end{eqnarray}$

where $r_L$ is the specific resistance in $[\Omega \, mm]$. Longitudinal resistance grows in proportion to length and shrinks in inverse propotion to cross-section.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Cable_theory_B.png" width="300">

(Slide-7)



# Longitudinal current and potential drop

Ohm's Law prescribes a correspondence between any longitudinal current $I_L$ and potential drop $\Delta V$

$\begin{eqnarray}
\Delta V = - R_L \, I_L  = -\frac{r_L\, \Delta x}{\pi\, a^2}\, I_L
\qquad
\Leftrightarrow
\qquad
I_L = -\frac{\pi\,a^2}{r_L} \, \frac{\Delta V}{\Delta x}
\end{eqnarray}$

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Cable_theory_B.png" width="300">

The *minus sign* follows from the convention that $\Delta V$ is positive when the potential increases with $x$, whereas $I_x$ is positive when flowing towards positive $x$.

In other words, current flows towards *decreasing* potential (i.e., downhill).

In the limit $\Delta x \rightarrow 0$, the expression for longitudinal current $I_L$ current becomes

$\begin{eqnarray}
 \qquad\qquad -\frac{\pi\,a^2}{r_L} \, \frac{\Delta V}{\Delta x} \quad \to \quad -\frac{\pi\,a^2}{r_L} \, \frac{\partial V}{\partial x}
\end{eqnarray}$

(Slide-8)

# Radial resistance

The resistance $R_m$ of the membrane surface of a cylindrical segment of length $\Delta x$, radius $a$, and circumference $2\pi\,a$

$\begin{eqnarray}
R_m = \frac{r_m}{2\pi\,a\,\Delta x}
\end{eqnarray}$

where $r_m$ is the specific membrane resistance in $[\Omega \, mm^2]$.  Radial resistance shrinks in inverse proportion to the membrane area, $2\pi \, a\, \Delta x$.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Cable_theory_C.png" width="250">

(Slide-9)

# Radial current and inside potential

For a passive cable, $r_m$ is constant, so that Ohm's Law relates the potential $V$ inside the cable segment (relative to outside) and the membrane current $i_m$

$\begin{eqnarray}
V = I_m \, R_m \qquad\Leftrightarrow\qquad V = 2\pi \, a \, \Delta x \, i_m \,  \frac{r_m}{2\pi\,a\,\Delta x} = i_m \, r_m
\end{eqnarray}$

where $I_m$ is the absolute membrane current (in $[A]$) and $i_m$ is the specific current (in $[A \, mm^{-2}]$). By convention, membrane currents are positive when flowing out of the segment.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Cable_theory_C.png" width="250">

(Slide-10)

# Net current and membrane capacitance

After considering all flows into and out of the cable segment, any net current charges the capacitance $C_m$ provided by membrane area $2\pi \, a \, \Delta x$:

$\begin{eqnarray}
I_{net} = C_m \, \frac{\partial V}{\partial t} \qquad\qquad C_m = 2\pi\,a\,\Delta x\, c_m
\end{eqnarray}$

where $c_m$ is the specific capacitance in $[F \, mm^{-2}]$. For the sake of generality, we also consider an electrode current delivered to the inside of our cable segment. By convention, electrode currents are positive when flowing into the segment.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Cable_theory_D.png" width="500">

(Slide-10)

# Cable equation I

In summary, there are four possible contributions to such a net current:


- through surface of cylinder
- longitudinally through left face
- longitudinally through right face
- through an electrode

$\begin{eqnarray}
\qquad\qquad I_\mathit{net} = I_\mathit{left} - I_\mathit{right} - I_\mathit{membrane} + I_\mathit{electrode}
\\
\\
\end{eqnarray}$

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Cable_theory_D.png" width="500">

(Slide-11)

# Cable equation II

Our final goal is to compute potential $V(x,t)$ as a function of position $x$ and time $t$. To this end, we relate the net current and each of the contributing currents to the potential $V(x,t)$:

$\begin{eqnarray}
I_\mathit{net} = I_\mathit{left} - I_\mathit{right} - I_\mathit{membrane} + I_\mathit{electrode}
\\
\end{eqnarray}$

Substitute each current by an expression with $V$:

$\begin{eqnarray}
2\pi a\Delta x \, c_m \frac{\partial V}{\partial t} =
-\left.\left(\frac{\pi\,a^2}{r_L}\,\frac{\partial V}{\partial x}\right)\right|_{l}
\,+\,
\left.\left(\frac{\pi\,a^2}{r_L}\,\frac{\partial V}{\partial x}\right)\right|_{r}
+ 2\pi a \Delta x \, ( -i_m + i_e )
\\
\end{eqnarray}$

Multiply with $r_m$ and divide by $2\pi a \Delta x$:

$\begin{eqnarray}
r_m \, c_m \, \frac{\partial V}{\partial t} = \frac{a \, r_m}{2 \, r_L}\,  \frac{1}{\Delta x} \, \left[ \left.\left(\frac{\partial V}{\partial x}\right)\right|_{r} \,-\,
\left.\,\left(\frac{\partial V}{\partial x}\right)\right|_{l}\right] - r_m \,  i_m + r_m \, i_e
\\
\end{eqnarray}$

Simplify by defining a **time-constant** $\tau_m = c_m \, r_m$ and a **length-constant** $\lambda_m = \sqrt{\frac{a \, r_m}{2 \, r_L}}$ for our cable:

$\begin{eqnarray}
\tau_m \,  \frac{\partial V}{\partial t} = \lambda_m^2 \, \frac{1}{\Delta x} \, \left[ \left.\left(\frac{\partial V}{\partial x}\right)\right|_{r} \,-\,
\left.\,\left(\frac{\partial V}{\partial x}\right)\right|_{l}\right] - V + r_m \, i_e
\end{eqnarray}$

(Slide-12)



# Cable equation III

From previous slide

$\begin{eqnarray}
\tau_m \,  \frac{\partial V}{\partial t} = \lambda_m^2 \, \frac{1}{\Delta x} \, \left[ \left.\left(\frac{\partial V}{\partial x}\right)\right|_{r} \,-\,
\left.\,\left(\frac{\partial V}{\partial x}\right)\right|_{l}\right] - V + r_m \, i_e
\end{eqnarray}$

$\begin{eqnarray}
\Delta x \rightarrow 0
\end{eqnarray}$

$\begin{eqnarray}
\qquad\qquad \tau_m \, \frac{\partial V}{\partial t} = \lambda_m^2 \, \frac{\partial^2 V}{\partial x^2} - V+ r_m \, i_e
\end{eqnarray}$

This partial differential equation is the **cable equation**.  It describes how currents entering, leaving, and flowing along the dendrite change the membrane potential.   Typically, the CE is solved for certain boundary conditions (as we will do in the next section).

(Slide-13)

# **3. Linear cable theory**

Linear cable theory assumes a cyclindrical dendrite with a **passive** membrane: without voltage-dependent conductances.

We consider potential $V(x,t)$ and membrane current $i_m(x,t)$ at location $x$ and time $t$ and its dependence on any input (electrode) currents $i_e(x,t)$. The situation is governed by

$\begin{eqnarray}
\qquad\qquad\qquad V(x,t) = r_m \, i_m(x,t)
\end{eqnarray}$

and by cable equation

$\begin{eqnarray}
\qquad\tau_m \, \frac{\partial V}{\partial t} = \lambda_m^2 \frac{\partial^2 V}{\partial x^2}\,- V+\, r_m \, i_e
\end{eqnarray}$

where $\tau_m = r_m \, c_m$ and $\lambda_m = \sqrt{\frac{a\,r_m}{2 r_L}}$ are time constant and length constant, respectively.

The constants depend on specific radial resistance $r_m$,  specific longitudinal resistance $r_L$, specific capacitance $c_m$, and radius $a$ of the cylindrical volume.

**Next, we consider analytical solutions for some special cases (boundary conditions).**

(Slide-15)


# Infinite cable with constant local current input

We inject a constant current $i_0$ at location $x=0$ of an infinitely long cable. What spatial distribution will the potential $V(s)$ assume at steady-state?

The two boundary conditions

$\begin{eqnarray}
i_e(x) = i_0 \, \delta(x), \qquad\qquad \frac{\partial V}{\partial t} = 0
\end{eqnarray}$

modify and simplify the cable equation:

$\begin{eqnarray}
\qquad \Rightarrow \qquad 0 = \lambda_m^2 \, \frac{\partial^2 V}{\partial x^2} - V+r_m i_0 \delta(x)
\end{eqnarray}$

We can guess the solution

$\begin{eqnarray}
V &=& \frac{R_\lambda \, i_0}{2} \, \exp\left(-\left|\frac{x}{\lambda_m}\right| \right)
\\
\frac{\partial V}{\partial x} &=& -sign(x) \, \frac{R_\lambda i_0}{2\lambda_m} \, \exp\left(-\left|\frac{x}{\lambda_m}\right| \right)
\\
\frac{\partial^2 V}{\partial x^2} &=& -2 \delta(x) V_0 + \frac{R_\lambda i_0}{2\lambda_m^2}  \exp\left(-\left|\frac{x}{\lambda_m}\right| \right)
\end{eqnarray}$

and substitute back to confirm and to obtain the constant

$\begin{eqnarray}
R_\lambda = \frac{r_m }{2 \pi a \lambda_m} = \frac{r_L \lambda_m}{\pi a^2}
\end{eqnarray}$

which is the effective input resistance of the infinite cable.

(Slide-16)

In [ ]:
# @title Steady-state for constant local current input {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact


# Define an interactive plot
def plot_infinite_cable(i_0 = 1, lambda_m = 2):
    """(Slide-17)"""

    # Get V0

    r_m = 8;
    V_0 = r_m * i_0 / (2 * lambda_m**2);

    # spatial range
    xi = np.linspace(-10,10,1001);

    # compute V
    Vi = V_0 * np.exp(-np.abs(xi/lambda_m));

    # compute dV/dx
    dVidx = -np.sign(xi) * V_0 * np.exp(-np.abs(xi/lambda_m));

    # Create a figure and a grid of subplots (2 rows, 1 columns)
    fig, axs = plt.subplots(1, 1, figsize=(8, 4))

    # First subplot
    plt.subplot(1, 1, 1)  # (nrows, ncols, plot_index)
    plt.plot(xi, dVidx, 'r--', linewidth=1)
    plt.plot(xi, Vi, 'b-', linewidth=1)
    plt.title('Constant input at x=0', fontsize=18)
    plt.xlabel('x', fontsize=14)
    plt.ylabel('Potential V(x)', fontsize=14)

    # Adjust layout to prevent overlap
    plt.tight_layout()

    # Display the plots
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_infinite_cable, i_0=(1,5,1), lambda_m=(0.5, 5, 0.5))



interactive(children=(IntSlider(value=1, description='i_0', max=5, min=1), FloatSlider(value=2.0, description=…

<function __main__.plot_infinite_cable(i_0=1, lambda_m=2)>

# Infinite cable with linear voltage

We consider an infinite cable in which the initial voltage changes linearly with position. How will this voltage profile decay over time?

Our boundary conditions
$\begin{eqnarray}
V_0(x) = \alpha_1\, x+\alpha_0 \quad \Leftrightarrow \quad \frac{\partial^2 V}{\partial x^2} = 0, \qquad i_e(x,t) = 0 \quad \forall x,t
\end{eqnarray}$

modify and simplify the cable equation

$\begin{eqnarray}
\qquad \Rightarrow \qquad  \tau_m \, \frac{\partial V}{\partial t} =  - V
\end{eqnarray}$

and we can immediately see the solution

$\begin{eqnarray}
\qquad V(x,t) = V_0(x) \, \exp\left(-\frac{t}{\tau_m}\right)
\end{eqnarray}$

(Slide-18)

In [ ]:
# @title Infinite cable with linear voltage {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact


# Define an interactive plot
def plot_linear_voltage(tau_m = 3):
    """(Slide-19)"""

    # spatial range
    xi = np.linspace(-10,10,1001);

    xm = np.linspace(-10,10,11);

    # temporal range in fine steps
    tj = np.linspace(0,10,101);

    # temporal range in coarse steps
    tn = np.linspace(0,10,6);

    # compute V0i
    alpha_1 = 0.1;
    V0i = alpha_1 * xi;
    V0m = alpha_1 * xm;

    # Create a figure and a grid of subplots (1 rows, 1 columns)
    fig, axs = plt.subplots(1, 2, figsize=(8, 4))

    # First subplot
    plt.subplot(1, 2, 1)  # (nrows, ncols, plot_index)
    for n in range(0,len(tn)):
        plt.plot(xi, V0i * np.exp(-tn[n]/tau_m), 'r--', linewidth=1)
    plt.plot(xi, V0i, 'b', linewidth=2)
    plt.xlabel('x', fontsize=14)
    plt.ylabel('V(x,t)', fontsize=14)
    plt.title('Different times', fontsize=18)

    # Second subplot
    plt.subplot(1, 2, 2)  # (nrows, ncols, plot_index)
    for m in range(0,len(xm)):
        plt.plot(tj, V0m[m] * np.exp(-tj/tau_m), 'r--', linewidth=1)

    plt.xlabel('t', fontsize=14)
    plt.ylabel('V(x,t)', fontsize=14)
    plt.title('Different positions', fontsize=18)

    # Adjust layout to prevent overlap
    plt.tight_layout()

    # Display the plots
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_linear_voltage, tau_m=(0.5,5,0.5))



interactive(children=(FloatSlider(value=3.0, description='tau_m', max=5.0, min=0.5, step=0.5), Output()), _dom…

<function __main__.plot_linear_voltage(tau_m=3)>

# Infinite cable with multiple local current inputs

As shown above, the steady-state potential evoked by a local current input $I_1 = \delta(x-x_1)$ at position $x_1$ is
$\begin{eqnarray}
V_1(x) = I_e  R_\lambda \exp\left(-\left|\frac{x-x_1}{\lambda_m}\right| \right)
\end{eqnarray}$

What will the steady-state potential be when local currents are injected at multiple points $x_1$, $x_2$, $x_3$, ... ?

The answer is easy because the cable equation is **linear**!

If $V_1(x)$, $V_2(x)$, and $V_3(x)$ are each steady-state solutions of the cable equation

$\begin{eqnarray}
\lambda_m^2 \frac{\partial^2 V_{1,2,3}(x)}{\partial x^2} =  V_{1,2,3}(x) - r_m \, I_{1,2,3}(x)
\end{eqnarray}$

then the sum $V_{sum} = V_1(x) + V_2(x) + V_3(x)$ of all three potentials is also a steady-state solution

$\begin{eqnarray}
\lambda_m^2 \frac{\partial^2 V_{sum}}{\partial x^2} =  V_{sum} - r_m \, I_{sum}
\end{eqnarray}$

where $I_{sum}(x) = I_1(x) + I_2(x) + I_3(x)$ is the sum of all three currents.

(Slide-20)

In [ ]:
# @title Steady-state for multiple current inputs {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact


# Define an interactive plot
def plot_multiple_local_inputs(i_2 = 0.5, x_2 = 4, i_3 = 1.5, x_3 = -6):
    """(Slide-21)"""

    # input resistance and length constant
    R_lambda = 1;
    lambda_m = 2;

    # first input is fixed)
    i_1 = 1;
    x_1 = 0;

    # spatial range
    xi = np.linspace(-10,10,1001);

    # individual steady-state solutions
    V_1 = R_lambda * i_1 * np.exp(-np.abs((xi-x_1)/lambda_m));
    V_2 = R_lambda * i_2 * np.exp(-np.abs((xi-x_2)/lambda_m));
    V_3 = R_lambda * i_3 * np.exp(-np.abs((xi-x_3)/lambda_m));

    # sum of individual solutions
    V_sum = V_1 + V_2 + V_3;

    # Create a figure and a grid of subplots (2 rows, 1 columns)
    fig, axs = plt.subplots(1, 1, figsize=(8, 4))

    # First subplot
    plt.subplot(1, 1, 1)  # (nrows, ncols, plot_index)
    plt.plot(xi, V_1, color=(0,0.5,0), linewidth=1)
    plt.plot(xi, V_2, color=(0.8,0,0), linewidth=1)
    plt.plot(xi, V_3, color=(0,0,1.0), linewidth=1)
    plt.plot(xi, V_sum, 'k', linewidth=2)
    plt.title('Multiple constant inputs', fontsize=18)
    plt.xlabel('x', fontsize=14)
    plt.ylabel('Potential V(x)', fontsize=14)

    # Adjust layout to prevent overlap
    plt.tight_layout()

    # Display the plots
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_multiple_local_inputs, i_2=(0.5,5,0.5), x_2=(-10, 10, 1), i_3=(0.5,5,0.5), x_3=(-10, 10, 1))



interactive(children=(FloatSlider(value=0.5, description='i_2', max=5.0, min=0.5, step=0.5), IntSlider(value=4…

<function __main__.plot_multiple_local_inputs(i_2=0.5, x_2=4, i_3=1.5, x_3=-6)>

# Infinite cable responds to transient local current pulse

We now consider the response of an infinite cable to a transient and local pulse of input current, specifically

$\begin{eqnarray}
i_e(x,t) = I_e\, \tau_m \, \delta(x) \delta(t)
\end{eqnarray}$


<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Current_pulse_C.png" width="500">

The cable equation

$\begin{eqnarray}
\tau_m \, \frac{\partial V}{\partial t} = \lambda_m^2 \frac{\partial^2 V}{\partial x^2}\,- V+\, r_m \, i_e
\end{eqnarray}$

has the solution (no proof, but you can check)

$\begin{eqnarray}
V(x, t) = \frac{I_e\, R_\lambda}{\sqrt{4\pi\,\lambda_m^2\,t/\tau_m}} \, \exp\left(-\frac{\tau_m\, x^2}{4 \lambda_m^2\, t} \right)
\, \exp\left(-\frac{t}{\tau_m} \right)
\end{eqnarray}$

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Current_pulse_A.png" width="500">

(Slide-22)


# Analysis of solution


$\begin{eqnarray}
V(x, t) = \frac{I_e\, R_\lambda}{\sqrt{4\pi\,\lambda_m^2\,t/\tau_m}} \, \exp\left(-\frac{\tau_m\, x^2}{4 \lambda_m^2\, t} \right)
\, \exp\left(-\frac{t}{\tau_m} \right)
\end{eqnarray}$

is the product of a time-dependent area $A(t)$ and a Gaussian spatial profile $G(x,\sigma)$ with variance $\sigma^2(t)$:

$\begin{eqnarray}
V(x,t) = A(t) \cdot G(x,\sigma)
\end{eqnarray}$

where the area decays exponentially over time

$\begin{eqnarray}
A(t) =  I_e\, R_\lambda \, \exp\left(-\frac{t}{\tau_m} \right)
\end{eqnarray}$

and the Gaussian profile

$\begin{eqnarray}
G(x,\sigma) = \frac{1}{\sqrt{2\pi\,\sigma^2}}\, \exp\left(-\frac{x^2}{2 \sigma^2} \right)
\end{eqnarray}$

that broadens with time.  Specifcially, its variance $\sigma^2(t)$ grows linearly with time

$\begin{eqnarray}
\sigma^2(t)= \frac{ 2 \, \lambda_m^2 }{ \tau_m} \, t
\end{eqnarray}$

(Slide-23)



In [ ]:
# @title Response to impulse: gaussian profile decays with time {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact

def GetColormap(name):
    clrmp = plt.get_cmap(name);
    # Define the number of colors (e.g., 256)
    num_colors = 256
    # Get the colormap as an array of RGB values
    clrmp_array = clrmp(np.linspace(0, 1, num_colors))
    return clrmp_array, num_colors

def GetColorindex( x, xmin, xmax, num_colors ):
    cix = round( (x-xmin)/(xmax-xmin) * (num_colors-1) );
    return cix

# Define an interactive plot
def plot_gaussian_profile_with_time(Delta_t = 0.5):
    """(Slide-24)"""

    # input resistance and length constant
    R_lambda = 1;
    lambda_m = 2;
    tau_m    = 1;
    I_e      = 1;

    # spatial range
    xi = np.linspace(-10,10,1001);

    # time steps
    tj = np.linspace(Delta_t,5*Delta_t,6);

    # potential profiles decaying with time
    sigma2_j = 2 * lambda_m**2 / tau_m * tj;

    xij, tij = np.meshgrid(xi, tj, indexing='ij')

    # compute area, variance, and gaussian profile
    area_ij = I_e * R_lambda * np.exp(-tij/tau_m);
    sigma2_ij = 2 * lambda_m**2 / tau_m * tij;
    gaussian_ij = 1 / np.sqrt(2 * np.pi * sigma2_ij) * np.exp(-xij**2 / (2 * sigma2_ij))

    # compute voltage
    V_ij = area_ij * gaussian_ij;

    Vmax = np.max(V_ij);

    # Create a figure and a grid of subplots (2 rows, 1 columns)
    fig, axs = plt.subplots(1, 1, figsize=(8, 4))

    # Get colormap
    (clrmp_array, num_colors) = GetColormap('jet');

    # First subplot
    plt.subplot(1, 1, 1)  # (nrows, ncols, plot_index)
    str1 = 't=';
    str2 = 'ms';
    for j in range(0,len(tj)):
        cix = GetColorindex(j, 0, len(tj), 256);
        clrj = clrmp_array[cix,:];
        plt.plot(xi, V_ij[:,j], '-', color=clrj, linewidth=1)
        str2 = f"{tj[j]:.2f}"
        plt.text(-10 + j*20/len(tj), 0.9*Vmax, str1+str2, color=clrj)
    plt.xlabel('position x', fontsize=14)
    plt.ylabel('V(x,t)', fontsize=14)

    # Adjust layout to prevent overlap
    plt.tight_layout()

    # Display the plots
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_gaussian_profile_with_time, Delta_t=(0.1,1,0.1) )



interactive(children=(FloatSlider(value=0.5, description='Delta_t', max=1.0, min=0.1), Output()), _dom_classes…

<function __main__.plot_gaussian_profile_with_time(Delta_t=0.5)>

# Local peak potential $V_{peak}$

To determine the time $t_\mathit{peak}$, at which location $x$ reaches its local peak potential $V_\mathit{peak}$, we set

$\begin{eqnarray}
0 &\stackrel{!}{=}& \frac{\partial V(x,t)}{\partial t} = G(x,\sigma) \frac{\partial A(t)}{\partial t}  + A(t) \frac{\partial G(x,\sigma)}{\partial \sigma} \dot\sigma =
\\
&=& \left[-\frac{1}{\tau_m} + \dot \sigma\left(-\frac{1}{\sigma} + \frac{x^2}{\sigma^3}\right) \right] V(x,t)=
\\
&=& \left[ - \frac{1}{\tau_m}   -\frac{1}{2t} + \frac{\tau_m x^2}{4 t^2 \lambda_m^2} \right] \, V(x,t) \stackrel{!}{=} 0
\end{eqnarray}$

where $\sigma(t) = \lambda_m \, \sqrt{\frac{2\,t}{\tau_m}}$ and $\dot\sigma(t) = \lambda_m \, \sqrt{\frac{1}{2 \, t \, \tau_m}}$ and $\frac{\dot \sigma}{\sigma}=\frac{1}{2t}$

Solving the quadratic expression gives

$\begin{eqnarray}
-\frac{t^2}{\tau_m^2} - \frac{t}{2\tau_m} -1 + \frac{x^2}{4\lambda_m^2} + 1= 0
\end{eqnarray}$

$\begin{eqnarray}
\Rightarrow \qquad \frac{t}{\tau_m} = -1 + \sqrt{\left( \frac{x}{2\lambda_m}\right)^2 + 1 } \qquad \Rightarrow \qquad t_\mathit{peak} \approx \frac{\tau_m}{2\lambda_m} x
\end{eqnarray}$

so that the 'propagation velocity' of the local peak potential is approximately

$\begin{eqnarray}
v_\mathit{peak} \equiv \frac{x}{t_\mathit{peak}} \approx 2 \, \frac{\lambda_m}{\tau_m}
\end{eqnarray}$

(Slide-25)



# Normalized potential

Normalizing the potential to its local maximum

$\begin{eqnarray}
V_\mathit{norm}(x,t) \equiv \frac{V(x,t)}{V_\mathit{peak}(x)}
\end{eqnarray}$

illustrates the propagation of the local peak potential:

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Current_pulse_B.png" width="500">

(Slide-26)

In [ ]:
# @title Response to impulse: propagation of peak potential {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact

def GetColormap(name):
    clrmp = plt.get_cmap(name);
    # Define the number of colors (e.g., 256)
    num_colors = 256
    # Get the colormap as an array of RGB values
    clrmp_array = clrmp(np.linspace(0, 1, num_colors))
    return clrmp_array, num_colors

def GetColorindex( x, xmin, xmax, num_colors ):
    cix = round( (x-xmin)/(xmax-xmin) * (num_colors-1) );
    return cix

def GetPeakPotential( x, V_0, lambda_m, tau_m ):

    t_peak = tau_m  * (-1 + np.sqrt(1+(x/(2*lambda_m))**2));

    area_t_peak = V_0 * np.exp(-t_peak/tau_m);

    sigma2_t_peak = 2 * lambda_m**2 * t_peak / tau_m;

    gaussian_t_peak = 1 / np.sqrt(2 * np.pi * sigma2_t_peak) * np.exp(-x**2 / (2 * sigma2_t_peak))

    # compute voltage
    V_x_peak = area_t_peak * gaussian_t_peak;

    return V_x_peak, t_peak

# Define an interactive plot
def plot_propagation_peak_potential(Delta_t = 0.5):
    """(Slide-24)"""

    # input resistance and length constant
    R_lambda = 1;
    lambda_m = 2;
    tau_m    = 1;
    I_e      = 1;

    # spatial range (without zero to avoid numerical issues)
    xi_neg = np.linspace(-10,-0.01,500);
    xi_pos = np.linspace(0.01,10,500);
    xi = np.concatenate((xi_neg, xi_pos));

    # time steps
    tj = np.linspace(Delta_t,5*Delta_t,6);

    # potential profiles decaying with time
    sigma2_j = 2 * lambda_m**2 / tau_m * tj;

    xij, tij = np.meshgrid(xi, tj, indexing='ij')

    # compute area, variance, and gaussian profile
    area_ij = I_e * R_lambda * np.exp(-tij/tau_m);
    sigma2_ij = 2 * lambda_m**2 / tau_m * tij;
    gaussian_ij = 1 / np.sqrt(2 * np.pi * sigma2_ij) * np.exp(-xij**2 / (2 * sigma2_ij))

    # compute voltage
    V_ij = area_ij * gaussian_ij;

    (Vpeak_i, t_peak) = GetPeakPotential( xi, R_lambda*I_e, lambda_m, tau_m )

    Vpeak_ij = np.tile(Vpeak_i.reshape(-1, 1), (1, 6))

    Vrel_ij = V_ij / Vpeak_ij

    Vmax_j = np.max(Vrel_ij, axis=0);

    # Create a figure and a grid of subplots (2 rows, 1 columns)
    fig, axs = plt.subplots(1, 1, figsize=(8, 4))

    # Get colormap
    (clrmp_array, num_colors) = GetColormap('jet');

    # First subplot
    plt.subplot(1, 1, 1)  # (nrows, ncols, plot_index)
    str1 = 't=';
    str2 = 'ms';
    for j in range(0,len(tj)):
        cix = GetColorindex(j, 0, len(tj), 256);
        clrj = clrmp_array[cix,:];
        plt.plot(xi, Vrel_ij[:,j], '-', color=clrj, linewidth=1)
        str2 = f"{tj[j]:.2f}"
        plt.text(-10 + j*20/len(tj), 0.2, str1+str2, color=clrj)
    plt.xlabel('position x', fontsize=14)
    plt.ylabel('V(x,t)/Vpeak(x)', fontsize=14)
    plt.title('Each position x normalized separately!', fontsize=18)

    # Adjust layout to prevent overlap
    plt.tight_layout()

    # Display the plots
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_propagation_peak_potential, Delta_t=(0.1,1,0.1) )



interactive(children=(FloatSlider(value=0.5, description='Delta_t', max=1.0, min=0.1), Output()), _dom_classes…

<function __main__.plot_propagation_peak_potential(Delta_t=0.5)>

# Footnote on radial and longitudinal resistance

It is instructive to compare the radial and longitudinal resistance for different cable lengths $L$

$\begin{eqnarray}
\qquad R_m(L) = \frac{r_m}{2\pi a L}, \qquad\qquad R_L(L) = \frac{r_L L}{\pi a^2}
\end{eqnarray}$

When cable length is $\lambda_m = \sqrt{\frac{a\,r_m}{2 r_L}}$, radial and longitudinal resistance are equal

$\begin{eqnarray}
\qquad\qquad\qquad\qquad R_m(\lambda_m) = R_L(\lambda_m)
\end{eqnarray}$



(Slide-27)

# Summary linear cable theory

- Analytical solutions of CE for certain special cases.

- Constant, local input current $\rightarrow$ exponential attenuation over space, with length constant $\lambda_m$.

- Multiple constant, local input currents $\rightarrow$ multiple exponential attenuations.

- Transient, local input current $\rightarrow$ Gaussian attenuation over space, exponential attenuation over time, with time-constant $\tau_m$.

- Local peak potential propagates with
$\begin{eqnarray}
v_{cond} = 2 \frac{\lambda_m}{\tau_m}
\end{eqnarray}$

- LCT offers analytical solutions for conduction delays, attenuation, summation and cancellation of current pulses!

(Slide-28)

# 4. Dendritic trees (optional)

To model complex dendritic trees, we must consider not only cylindrical dendrite segments, but also dendrites with increasing width and dendrites with branching points.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Dendritic_trees_A.png" width="300">

(Slide-29)





# Effect of branching

To illustrate the effect of branching, we consider a T-junction of 3 semi-infinite cables, with radii $a_1$, $a_2$, $a_3$.   We measure distance $x$ outward from the junction and inject a current into branch $2$ at distance $x_e$.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Dendritic_trees_B.png" width="400">

The static current into branch $2$ evokes the following steady-state voltages in branches $1$, $2$, and $3$:

$\begin{eqnarray}
v_1(x) &=&  I_e\, R_{\lambda_1} \, p_1 \, \exp\left( -\frac{x}{\lambda_1} - \frac{x_e}{\lambda_2} \right)
\\
\\
v_2(x) &=&  I_e\,R_{\lambda_2} \, \left[ (p_2-1) \, \exp\left(-\frac{x}{\lambda_2} -\frac{x_e}{\lambda_2} \right) \, + \,
  \exp\left( -\frac{|x_e-x|}{\lambda_2} \right)\right]
\\
\\
v_3(x) &=&  I_e\, R_{\lambda_3} \, p_3 \, \exp\left( -\frac{x}{\lambda_3} - \frac{x_e}{\lambda_2} \right)
\end{eqnarray}$

Exponential attenuation within branches is governed by length constants $\lambda_i$ (``electrotonic length'').  Transmission between branches is governed by  input resistances $R_{\lambda_i}$ and coupling factors $p_i$.

The illustration shows attenuation 'thick-to-thin' (left) and 'thin-to-thick' (right):

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Dendritic_trees_C.png" width="500">

D&A Fig. 6-9

(Slide-30)

# Effect of branch thickness

Branch thickness increases the length constant and decreases the input resistance:

$\begin{eqnarray}
\lambda_m = \sqrt{\frac{a\,r_m}{2 r_L}}, \qquad\qquad R_\lambda = \frac{r_m }{2 \pi a \lambda_m} = \frac{r_L \lambda_m}{\pi a^2}
\end{eqnarray}$

In addition, it has a pronounced effect on coupling factors:

$\begin{eqnarray}
\qquad\qquad p_i = \frac{a_i^{3/2}}{a_1^{3/2} + a_2^{3/2} + a_3^{3/2}}
\end{eqnarray}$

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Dendritic_trees_B.png" width="400">

(Slide-31)

# Equivalent cable

Rall (1957, 1977) has suggested a method for applying linear cable theory to responses of real neurons.  It allows us to compute the effect of the soma potential of synaptic input currents on the distal dendrite.  

Assuming certain conditions are met, notably certain diameter proportions at branching points $1\to 2,3$

$\begin{eqnarray}
a_1^{3/2} = a_2^{3/2} + a_2^{3/2}  \qquad\Leftrightarrow\qquad  p_1= p_2 + p_3  = \frac{1}{2}
\end{eqnarray}$

the entire dendritic tree between soma and synapse can be replaced by single **equivalent cable**.  

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Dendritic_trees_E.png" width="500">

(Slide-32)

# Constant current to soma


The Rall model provides us with *equivalent circuits*. Consider a constant current $I_e$ injected into the soma.  

The voltage measured at a distance $x$ is given by an
equivalent circuit with three resitances, $R_\mathit{soma}$ (soma to ground), $R_1$ (soma to point $x$), and $R_2$ (point $x$ to ground).

$\begin{eqnarray}
V_\mathit{soma} = \frac{(R_1+R_2) R_\mathit{soma}}{R_1+R_2 +R_\mathit{soma}} \, I_e, \qquad\qquad V(x) = \frac{R_2 R_\mathit{soma}}{R_1+R_2 +R_\mathit{soma}} \, I_e
\end{eqnarray}$

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Equivalent_dendrite_A.png" width="500">

D&A Fig. 6-10

Formulas for $R_1$ and $R_2$ are provided by D&A, Chapter 6.

(Slide-33)

# Constant current to dendrite

Similarly, consider a constant current $I_e$ injected to the dendrite, with the soma potential clamped to $V_\mathit{rest}$.   The current entering the soma is given by an equivalent circuit with two resistances $R_3$ and $R_4$.  (The soma resistance is irrelevant because the soma potential is fixed.)

$\begin{eqnarray}
I_\mathit{soma} = \frac{R_4}{R_3+R_4} \, I_e, \qquad\qquad V(x) = \frac{R_3 R_4}{R_3+R_4} \, I_e
\end{eqnarray}$

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Equivalent_dendrite_B.png" width="500">

D&A Fig. 6-12

Formulas for $R_3$ and $R_4$ are provided by D&A, Chapter 6.

(Slide-34)

# Voltage and current attenuation

Rall's equivalent cable highlights a general result of linear cable theory (for constant current/voltage):


**Inward attenuation of current equals outward attenuation of voltage.**

For example, the dendrite-to-soma attenuation of current in D&A Fig. 6-10 equals the soma-to-dendrite attentuation of voltage in D&A Fig. 6-12 (see above).

(Slide-35)

# Summary dendritic trees

- Cable theory applies also to more complex dendrites (varying diameters, branching points).

- Voltage attenuation is governed by length constants (electrotonic length).

- Current distribution between branches is governed by coupling factors (diameter$^{3/2}$).

- The Rall model replaces an entire dendritic tree with an equivalent cable, which matches both the surface area and the electrotonic length of the original dendritic tree

- For steady-state currents and voltages, inward attenuation of current equals outward attenuation of voltage.

(Slide-36)

# **5. Morphoelectronic transform**

Neuronal morphology can be reconstructed in detail, for example from serial slices in the electronmicroscope (EM). After the arborizations (axonal and dendritic) have been defined as three dimensional membrane surfaces, the electric properties can be simulated numerically. This is done by apportioning the continuous dendrites into discrete compartments, each assumed to be electronically compact and represented by a single $V(x,t)$.   The number of compartments can range from one to several thousand, depending the desired accuracy.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Morphoelectronic_A.png" width="600">

D&A Fig. 6-15

(Slide-37)

# World GDP, in local purchasing power parity.


Here the familiar world map is distorted according to the purchasing power of each nation's population (in local currency, hence 'purchasing power parity' = PPP).

In global currency, the differences would appear even more extreme.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Worldmap_GDP_localPPP.png" width="600">

[worldmapper.org](https://www.worldmapper.org)

Zador and colleagues used similar distortions to illustrate the electric properties of neurons with realistic morphology.

They called this a **morphoelectrotonic transform**.

[Zador, Agmon-Snir, Segev (1995) The morphoelectrotonic transform: a graphical approach to dendritic function. J. Neurosci.](https://www.jneurosci.org/content/jneuro/15/3/1669.full.pdf)

(Slide-38)

# Logarithmic voltage attenuation

The morphoelectronic transform distorts the anatomical shape of a neuron into an 'electric shape' that illustrates **additive** electrical properties.


For example, a suitable electrical quantity is logarithmic voltage attenuation:

$\begin{eqnarray}
\frac{v(x_1)}{v(x_3)} =  \frac{v(x_1)}{v(x_2)} \cdot \frac{v(x_2)}{v(x_3)}
\\
\\
\ln\frac{v(x_1)}{v(x_3)} =  \ln\frac{v(x_1)}{v(x_2)} \,+\, \ln\frac{v(x_2)}{v(x_3)}
\end{eqnarray}$

*Note that log attenuation is additive!*

(Slide-39)

# Conduction delay

A second suitable quantity is the conduction delay for a voltage pulse.  

$\begin{eqnarray}
\Delta t_{13} = \Delta t_{12} + \Delta t_{23}
\qquad\qquad
\Delta t_{ij} = t_i - t_j
\end{eqnarray}$

The 'delay' can be defined, for example, for the centroid of the time-varying voltage:

$\begin{eqnarray}
t_i = \frac{\int \, t \, v(x_i,t) \, dt}{\int \, v(x_i,t) \, dt}
\end{eqnarray}$

*Conduction delays are also additive!*

(Slide-40)

# Inward attenuation

Inward attenuation and delay of a peripheral constant current and voltage pulse.   Note that apical and basal dendrites have comparable electric properties (cortical L5 pyramid of cat).

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Morphoelectronic_B.png" width="200">
<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Morphoelectronic_C.png" width="550">

The two small insets show attenuation and delay in the **outward** direction, to scale!

(Slide-41)



# Outward attenuation

Outward attenuation of alternating soma current.  Low-frequency signals penetrate much further than high-frequency signals (e.g., action potentials).  In general, outward attenuation is smaller than inward attenuation, thanks to the diminishing cross-section of dendrites.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Morphoelectronic_D.png" width="700">

D&A Fig. 6-14

(Slide-42)

# Electric shape of Layer 5 Pyramidal cell

Anatomical cell shape (left), electrontonic attenuation in units of 0.1 length constants (middle), and the same again rescaled to anatomical size (right).

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Morphoelectronic_I.png" width="700">

$\qquad\qquad\qquad\qquad$|-----| 100 $\mu m\qquad\qquad\qquad$ |-----| 0.1 $\lambda$

Zador et al. (1995)

(Slide-43)

# Electric shape of Layer 2/3 Pyramidal cell

Anatomical cell shape (left), electrontonic attenuation in units of 0.1 length constants (middle), and the same again rescaled to anatomical size (right).

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Morphoelectronic_G.png" width="700">

$\qquad\qquad\qquad\qquad$|-----| 100 $\mu m\qquad\qquad\qquad$|-----| 0.1 $\lambda$

Zador et al. (1995)

(Slide-44)

# Electric shape of cerebellar Purkinje cell

Anatomical cell shape (left), electrontonic attenuation in units of 0.1 length constants (middle), and the same again rescaled to anatomical size (right).

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Morphoelectronic_F.png" width="700">

$\qquad\qquad\qquad\qquad$|-----| 100 $\mu m\qquad\qquad\qquad$ |-----| 0.1 $\lambda$

Zador et al. (1995)

(Slide-45)

# Electric shape of CA1 Pyramidal cell

Anatomical cell shape (left), electrontonic attenuation in units of 0.1 length constants (middle), and the same again rescaled to anatomical size (right).


<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Morphoelectronic_E.png" width="700">

$\qquad\qquad\qquad\qquad\qquad$|-----| 100 $\mu m\qquad\qquad\qquad$ |-----| 0.1 $\lambda$

Zador et al. (1995)

(Slide-46)

# Summary morphoelectronic transform

- Detailed numerical simulation of complex dendrites (reconstructed from EM) reveals the `electric shape' of neurons.

- Outward attenuation depends on frequency.  Slow signals (resting potential) penetrate much further than fast signals (action potentials).

- The 'electric shape' of a neuron can differ dramatically from its anatomical shape (compact Purkinje, elongated CA1 pyramid, isotropic L3 pyramid).

(Slide-47)

# **6. Myelinated axons (advanced)**

We return to active membranes and action potential conduction. Unmyelinated axons can be modelled with multiple compartments, with a Hodgkin-Huxley model governing the membrane conductances of each compartment


<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Myelin_A.png" width="600">

D&A Fig. 6-17

(Slide-48)

# Scaling law for unmyelinated axons: $v\propto\sqrt{a}$

- Conduction velocities of unmyelinated axons range fom $0.4 \, m/s$ to $10 \, m/s$.


- Conduction velocity is proportional to the ratio between length constant (electrotonic length) and membrane time constant:

$\begin{eqnarray}
v_{cond} \propto \frac{\lambda_m}{\tau_m} = \frac{\sqrt{\frac{a \, r_m}{2 \, r_L}}}{r_m\, c_m} \propto \sqrt{a}
\end{eqnarray}$

- The square root dependence on axon radius means that very thick axons are required to achieve high propagation speeds.  

- The squid giant axon ($800\,\mu m$) conducts $20\times$ faster than a typical (unmyelinated) mammalian axon ($2\,\mu m$), but at the expense of a $400\times$ larger radius and $16 000\times$ times more mass.

(Slide-49)


# Myelinated axon

Many vertebrate axons are covered by an insulating layer of myelin, except at periodic gaps, called the nodes of Ranvier, where the membrane contains a high density of voltage-dependent channels.  The myelin consists of many layers of glial membrane wrapped around the axon.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Myelin_C.png" width="600">

D&A Fig. 6-18

(Slide-50)

# Active and passive segments

A myelinated axon can be modeled as a series of compartments, alternatingly between **active membrane** (Ranvier node) and **passive membrane** with high membrane resistance (myelinated segment).

An equivalent circuit for a myelinated axon is shown below.  Myelinated segments are **passive** and represented by capacitance, longitudinal resistance, and (near infinite) radial resistance. Nodes of Ranvier are **active** and additionally contain voltage-dependent conductances.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Myelin_B.png" width="600">

D&A Fig. 6-18

(Slide-51)

# Capacity of myelinated segment

To compute capacitance of a myelin-covered axon,  we take the myelin to consist of a concentric arrangement of thin individual shells.  Note that inverse capacitances add when arranged in series:

$\begin{eqnarray}
\frac{1}{C_i} = \frac{1}{2\pi \, c_m \,  a_i \, L}
\qquad\qquad\qquad
\frac{1}{C_m} = \sum_i \, \frac{1}{C_i} = \sum_i \, \frac{1}{2\pi \, c_m \,  a_i \, L}
\end{eqnarray}$

with specific capacitance $c_m$, length $L$, radius $a_i=i \cdot d_m$, and individual thickness $d_m$.  

Instead of working out this sum, it is easier to take the limit $\Delta a \rightarrow 0$ and to integrate from interior radius $a_1$ to exterior radius $a_2$:

$\begin{eqnarray}
\frac{1}{C_m} = \frac{1}{2\pi \, c_m \,  d_m \, L} \, \int_{a_1}^{a_2} \, \frac{da}{a} = \frac{1}{2\pi \, c_m \, d_m \, L} \, \ln\frac{a_2}{a_1}
\end{eqnarray}$

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Myelin_C.png" width="300">

(Slide-52)

# Cable equation for $r_m=\infty$

When we re-drive the cable equation in the special case of infinite resistance and zero membrane current, we obtain

$\begin{eqnarray}
r_m\rightarrow\infty, i_m\rightarrow 0\qquad\Rightarrow\qquad C_m \, \frac{\partial V}{\partial t} = \frac{\pi \, L \, a_1^2}{r_L}\,  \frac{\partial^2 V}{\partial x^2}
\end{eqnarray}$

or

$\begin{eqnarray}
\qquad\qquad\frac{\partial V}{\partial t} = D \, \frac{\partial^2 V}{\partial x^2} \qquad\qquad D =  \frac{\pi\, L \,  a_1^2}{C_m \, r_L}
\end{eqnarray}$

where $D$ is the 'diffusion constant'. When we substitute the expression for $\frac{1}{C_m}$ (in terms of $a_1$ and $a_2$) derived above, this becomes

$\begin{eqnarray}
D =  \frac{\pi\, L \,  a_1^2}{C_m \, r_L}= \frac{\pi \, L \, a_1^2}{2\pi \, c_m \, d_m \, L \, r_L} \, \ln\frac{a_2}{a_1}= \frac{ a_1^2}{2 \, c_m \, d_m \, r_L} \, \ln\frac{a_2}{a_1}
\end{eqnarray}$

Let's try to understand some implications!

(Slide-53)

# Diffusion constant and conduction velocity

The diffusion constant is proportional to the square of the conduction velocity:

$\begin{eqnarray}
v_{cond}^2 \propto D =  \frac{ 1}{2 \, c_m \, d_m \, r_L} \, a_1^2 \, \ln\frac{a_2}{a_1}
\end{eqnarray}$

For a given outer radius $a_2$, the diffusion constant (and conduction velocity) is a non-monotonic function of inner radius $a_1$:



<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec08_Myelin_D.png" width="500">

(Slide-54)

# Optimal inner radius

For a given outer radius $a_2$, the maximal values of $D$ and $v_{cond}$ are reached for

$\begin{eqnarray}
0 \stackrel{!}{=} \frac{dD}{da_1} \propto  2a_1 \, \ln\frac{a_2}{a_1} - a_1^2 \, \frac{1}{a_1} = a_1 \, \left(2 \, \ln\frac{a_2}{a_1} - 1 \right)
\end{eqnarray}$

$\begin{eqnarray}
\qquad\qquad \Rightarrow
\end{eqnarray}$

$\begin{eqnarray}
a_1 = a_2 \, \exp(-1/2) \approx 0.6 \, a_2
\end{eqnarray}$


A fractional inner radius of $\sim 0.6$ is typical for myelinated axons!

Conduction velocities of myelinated axons reach up to $150\,m/s$.

(Slide-55)

# Scaling law for myelinated axon: $v\propto a$

At the optimal inner radius $a_1$, the propagation velocity becomes proportional to $a_2$

$\begin{eqnarray}
D &\propto& a_1^2 \, \ln\frac{a_2}{a_1}, \qquad\qquad a_1 = a_2 \exp(-1/2), \qquad\qquad \ln \frac{a_2}{a_1} = \frac{1}{2}
\end{eqnarray}$


$\begin{eqnarray}
\qquad\qquad \Rightarrow \qquad\qquad\qquad\qquad D &\propto& a_2^2
\end{eqnarray}$

We conclude that the conduction velocity is proportional to axon radius $a$ (not $\sqrt{a}$ !)

$\begin{eqnarray}
\qquad\qquad\qquad\qquad v_{cond} \quad \propto \quad \sqrt{D} \quad \propto \quad a_2
\end{eqnarray}$

Myelination increases velocity and establishes a favorable scaling law!

(Slide-56)

# Summary myelinated axons

- Unmyelinated axons can be modeled numerically as a series of compartments with active conductances.

- Unmyelinated axons conduct up to $10\,m/s$, velocity scaling with $\sqrt{a}$.


- Myelinated axons can be modeled as a series of compartments with, alternatingly, active and passive conductances.

- Myelination changes the physics of passive compartments, increasing $c_m$ and reducing conductance $1/r_m$ to zero.

- Myelinated axons conduct up to $150 \, m/s$, velocity scaling with $a$.

- In an optimal myelin sheath, inner diameter is $60\%$ of outer diameter.

(Slide-57)

# **Overview**

1. Motivation
2. Cable equation
3. Some solutions of linear cable theory
4. Dendritic trees (optional)
5. Morphoelectronic transform
6. Myelinated axons (advanced)

# **Next lecture:**

# **9. Receptive fields and reverse correlation**